In [0]:
spark.sql("DROP TABLE IF EXISTS merge_schema")

df_inicial = spark.createDataFrame([
    (1, "Ana"),
    (2, "Juan")
], ["id", "nombre"])

df_inicial.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("merge_schema")

# Validar el schema
spark.table("merge_schema").printSchema()




In [0]:
# Merge
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "merge_schema")

delta_table.alias("t").merge(df_nuevo.alias("s"), "t.id = s.id") \
 .whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [0]:
%skip
# Agregar con mergeSchema
df_nuevo = spark.createDataFrame([
    (1, "Ana X", 20),
    (2, "Juan C", 15),
    (3, "Pedro", 30),
    (4, "Maria", 25),
    (5, "Carolina", 25),
    (6, "Reinerys", 22)
], ["id", "nombre", "edad"])

df_nuevo.printSchema()

# Insertar el nuevo registro
df_nuevo.write \
    .format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable("merge_schema")

In [0]:
%sql
select * from merge_schema
order by id